# Add Date to Gold

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType,
)

In [0]:
from pyspark import pipelines as dp

## Variables


##Schema Definition

In [0]:
schema = StructType(
    [
        StructField(
            name="id_datum",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Datum als Nummer"},
        ),
        StructField(
            name="datum_date",
            dataType=DateType(),
            nullable=False,
            metadata={"comment": "Datum als Date"},
        ),
        StructField(
            name="wochentag_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Wochentag als Nummer"},
        ),
        StructField(
            name="wochentag_bez",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Wochentag ausgeschrieben"},
        ),
        StructField(
            name="wochentag_kbez",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Wochentag in Kurz"},
        ),
        StructField(
            name="tag_im_monat_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Tag im Monat"},
        ),
        StructField(
            name="tag_im_jahr_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Tag im Jahr"},
        ),
        StructField(
            name="tage_seit_1900",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Tage seit 1900"},
        ),
        StructField(
            name="jahr_kw_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Jahr und KW Nummer"},
        ),
        StructField(
            name="kw_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Aktuelle KW Nummer"},
        ),
        StructField(
            name="kw_bez",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Kurzbezeichnung der KW"},
        ),
        StructField(
            name="jahr_monat_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Jahr und Monat als Nummer"},
        ),
        StructField(
            name="monat_bez",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Monat ausgeschrieben"},
        ),
        StructField(
            name="monat_kbez",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Monat Beschreibung in Kurz"},
        ),
        StructField(
            name="monatjahr_kbez",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Monat und Jahr in Kurz"},
        ),
        StructField(
            name="monat_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Monat als Nummer"},
        ),
        StructField(
            name="jahr_quartal_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Jahr und Quartal Nummer (YYYYQQ)"},
        ),
        StructField(
            name="quartal_bez",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Quartal als Beschreibung mit Q1"},
        ),
        StructField(
            name="quartal_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Quartal als Nummer"},
        ),
        StructField(
            name="jahr",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Jahr"},
        ),
        StructField(
            name="islastofmonth",
            dataType=BooleanType(),
            nullable=False,
            metadata={"comment": "1 wenn es sich um den letzten Monat handelt"},
        ),
        StructField(
            name="time_to_end_of_month",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Anzahl Tage bis zum Ende des Monats"},
        ),
        StructField(
            name="time_to_end_of_year",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Anzahl Tage bis zum Ende des Jahres"},
        ),
    ]
)

##ETL

In [0]:
@dp.materialized_view(
    # Name der Zieltabelle
    name="gold.datum",
    # Beschreibung der Tabelle
    comment=f"Datum Tabelle",
    # Liquid Clustering (Statt partitioning und Z-Order)
    cluster_by=[],
    cluster_by_auto=True,
    # Beschreibung des Schemas
    schema=schema,
)
def common_dim_datum():
    df = spark.read.table(f"analytics.silver.calendar_stm_datum")

    schema_columns = [(field.name, field.dataType) for field in schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])
    return df